<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/global_risk_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# CELL 1 — GOOGLE DRIVE + TRAINING PATH SETUP
# GLOBAL RISK GATE
# =========================================================

from google.colab import drive

import os


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = (
    "/content/drive"
)

drive.mount(
    DRIVE_MOUNT,
    force_remount=False,
)

assert os.path.exists(
    f"{DRIVE_MOUNT}/MyDrive"
), "Google Drive belum mounted"


# =========================================================
# 2. GLOBAL RISK GATE ROOT
# =========================================================

SAVE_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/"
    "global_risk_gate"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)


# =========================================================
# 3. SUBDIRECTORIES
# =========================================================

RAW_DIR = (
    f"{SAVE_DIR}/raw"
)

FEATURE_DIR = (
    f"{SAVE_DIR}/features"
)

TEACHER_DIR = (
    f"{SAVE_DIR}/teacher"
)

TRUSTED_DIR = (
    f"{SAVE_DIR}/trusted"
)

PROCESSED_DIR = (
    f"{SAVE_DIR}/processed"
)

MODEL_DIR = (
    f"{SAVE_DIR}/models"
)

CHECKPOINT_DIR = (
    f"{MODEL_DIR}/checkpoints"
)

ONNX_DIR = (
    f"{MODEL_DIR}/onnx"
)

REPORT_DIR = (
    f"{SAVE_DIR}/reports"
)

BLIND_DIR = (
    f"{SAVE_DIR}/blind"
)


DIRECTORIES = [

    RAW_DIR,

    FEATURE_DIR,

    TEACHER_DIR,

    TRUSTED_DIR,

    PROCESSED_DIR,

    MODEL_DIR,

    CHECKPOINT_DIR,

    ONNX_DIR,

    REPORT_DIR,

    BLIND_DIR,
]


for directory in DIRECTORIES:

    os.makedirs(
        directory,
        exist_ok=True,
    )


# =========================================================
# 4. MODEL 01 — ACTION CLASSIFIER
# =========================================================

ACTION_CLASSIFIER_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/"
    "action_classifier"
)

ACTION_ONNX_PATH = (
    f"{ACTION_CLASSIFIER_DIR}/"
    "models/onnx/"
    "action_classifier.onnx"
)


# =========================================================
# 5. MODEL 02 — COMMAND RISK
# =========================================================

COMMAND_RISK_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/"
    "command_risk_v2"
)

COMMAND_RISK_ONNX_PATH = (
    f"{COMMAND_RISK_DIR}/"
    "models/onnx/"
    "command_risk_v2.onnx"
)


# =========================================================
# 6. DATASET PATHS
# =========================================================

RAW_CASES_PATH = (
    f"{RAW_DIR}/"
    "global_risk_raw_cases_v1.jsonl"
)

INFERENCE_FEATURES_PATH = (
    f"{FEATURE_DIR}/"
    "global_risk_features_v1.jsonl"
)

TEACHER_OUTPUT_PATH = (
    f"{TEACHER_DIR}/"
    "global_risk_teacher_v1.jsonl"
)

TEACHER_SAFEPOINT_PATH = (
    f"{TEACHER_DIR}/"
    "global_risk_teacher_safepoint_v1.json"
)

TEACHER_REJECTED_PATH = (
    f"{TEACHER_DIR}/"
    "global_risk_teacher_rejected_v1.jsonl"
)

TRUSTED_DATASET_PATH = (
    f"{TRUSTED_DIR}/"
    "global_risk_dataset_v1.jsonl"
)


# =========================================================
# 7. TRAIN / VAL / TEST PATHS
# =========================================================

TRAIN_PATH = (
    f"{PROCESSED_DIR}/"
    "train_v1.jsonl"
)

VAL_PATH = (
    f"{PROCESSED_DIR}/"
    "validation_v1.jsonl"
)

TEST_PATH = (
    f"{PROCESSED_DIR}/"
    "test_v1.jsonl"
)


# =========================================================
# 8. MODEL OUTPUT PATHS
# =========================================================

MODEL_PATH = (
    f"{MODEL_DIR}/"
    "global_risk_gate_v1.joblib"
)

ONNX_MODEL_PATH = (
    f"{ONNX_DIR}/"
    "global_risk_gate.onnx"
)

METRICS_PATH = (
    f"{REPORT_DIR}/"
    "metrics_v1.json"
)

ERROR_ANALYSIS_PATH = (
    f"{REPORT_DIR}/"
    "error_analysis_v1.csv"
)

BLIND_RESULTS_PATH = (
    f"{BLIND_DIR}/"
    "blind_results_v1.jsonl"
)


# =========================================================
# 9. SOURCE VALIDATION
# =========================================================

print()

print("=" * 78)
print("GLOBAL RISK GATE — STORAGE SETUP")
print("=" * 78)

print()

print(
    "Save directory       :",
    SAVE_DIR
)

print(
    "Action ONNX          :",
    ACTION_ONNX_PATH
)

print(
    "Command Risk ONNX    :",
    COMMAND_RISK_ONNX_PATH
)

print()

print(
    "Action model exists  :",
    os.path.exists(
        ACTION_ONNX_PATH
    )
)

print(
    "Risk model exists    :",
    os.path.exists(
        COMMAND_RISK_ONNX_PATH
    )
)

print()

print(
    "Teacher output       :",
    TEACHER_OUTPUT_PATH
)

print(
    "Teacher safe point   :",
    TEACHER_SAFEPOINT_PATH
)

print(
    "Trusted dataset      :",
    TRUSTED_DATASET_PATH
)

print()

print("=" * 78)

In [ ]:
# =========================================================
# CELL 2 — LABELS + FEATURE SCHEMA + SAFE POINT
# GLOBAL RISK GATE
# =========================================================

import os
import json
import time
import hashlib


# =========================================================
# 1. GLOBAL RISK LABELS
# =========================================================

RISK_GATE_LABELS = [

    "SAFE",

    "REVIEW",

    "SANDBOX",

    "BLOCK",
]


LABEL_TO_ID = {

    label:
        index

    for index, label
    in enumerate(
        RISK_GATE_LABELS
    )
}


ID_TO_LABEL = {

    index:
        label

    for label, index
    in LABEL_TO_ID.items()
}


# =========================================================
# 2. ACTION CLASSIFIER LABELS
# =========================================================

ACTION_LABELS = [

    "READ",

    "WRITE",

    "DELETE",

    "EXECUTE",

    "NETWORK",

    "INSTALL",

    "PRIVILEGED",

    "SYSTEM_CHANGE",
]


# Backward compatibility with parser / previous cells.

LABELS = ACTION_LABELS


# =========================================================
# 3. COMMAND RISK LABELS
# =========================================================

COMMAND_RISK_LABELS = [

    "BENIGN",

    "SUSPICIOUS",

    "DESTRUCTIVE",
]


# =========================================================
# 4. CONTEXT ENUMS
# =========================================================

TOOL_TYPES = [

    "shell",

    "python",

    "package_manager",

    "filesystem",

    "network",

    "service",

    "security_tool",

    "unknown",
]


TARGET_CONTEXTS = [

    "local_user",

    "local_system",

    "container",

    "sandbox",

    "remote_host",

    "unknown",
]


# =========================================================
# 5. SAMPLE ID
# =========================================================

def make_case_id(data):

    canonical = json.dumps(
        data,
        sort_keys=True,
        ensure_ascii=False,
        separators=(
            ",",
            ":"
        ),
    )

    digest = hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()

    return digest[:20]


# =========================================================
# 6. JSONL APPEND
# =========================================================

def append_jsonl(
    path,
    item,
):

    with open(
        path,
        "a",
        encoding="utf-8",
    ) as f:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False,
            )
        )

        f.write(
            "\n"
        )

        # Important for Colab / Drive safe-pointing.

        f.flush()

        os.fsync(
            f.fileno()
        )


# =========================================================
# 7. LOAD JSONL
# =========================================================

def load_jsonl(
    path,
):

    if not os.path.exists(
        path
    ):

        return []

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = (
                line.strip()
            )

            if not line:

                continue

            rows.append(
                json.loads(
                    line
                )
            )

    return rows


# =========================================================
# 8. LOAD COMPLETED CASE IDs
# =========================================================

def load_completed_case_ids():

    completed = set()

    if not os.path.exists(
        TEACHER_OUTPUT_PATH
    ):

        return completed

    with open(
        TEACHER_OUTPUT_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = (
                line.strip()
            )

            if not line:

                continue

            try:

                row = json.loads(
                    line
                )

                case_id = row.get(
                    "case_id"
                )

                if case_id:

                    completed.add(
                        case_id
                    )

            except Exception:

                # Ignore damaged final line,
                # do not destroy valid rows.

                continue

    return completed


# =========================================================
# 9. SAFE POINT
# =========================================================

def save_teacher_safepoint(
    processed,
    accepted,
    rejected,
    last_case_id=None,
):

    state = {

        "processed":
            int(processed),

        "accepted":
            int(accepted),

        "rejected":
            int(rejected),

        "last_case_id":
            last_case_id,

        "updated_at":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
    }


    tmp_path = (
        TEACHER_SAFEPOINT_PATH
        + ".tmp"
    )


    with open(
        tmp_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            state,
            f,
            indent=2,
            ensure_ascii=False,
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


    os.replace(
        tmp_path,
        TEACHER_SAFEPOINT_PATH,
    )


# =========================================================
# 10. LOAD SAFE POINT
# =========================================================

def load_teacher_safepoint():

    if not os.path.exists(
        TEACHER_SAFEPOINT_PATH
    ):

        return {

            "processed":
                0,

            "accepted":
                0,

            "rejected":
                0,

            "last_case_id":
                None,
        }


    with open(
        TEACHER_SAFEPOINT_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(
            f
        )


# =========================================================
# 11. STATUS
# =========================================================

existing_completed = (
    load_completed_case_ids()
)

existing_state = (
    load_teacher_safepoint()
)


print()

print("=" * 78)
print("GLOBAL RISK GATE — SCHEMA + SAFE POINT")
print("=" * 78)

print(
    "Gate labels        :",
    RISK_GATE_LABELS
)

print(
    "Action labels      :",
    len(
        ACTION_LABELS
    )
)

print(
    "Risk labels        :",
    COMMAND_RISK_LABELS
)

print()

print(
    "Completed teacher  :",
    len(
        existing_completed
    )
)

print(
    "Safe-point state   :",
    existing_state
)

print()

print("=" * 78)

In [ ]:
# =========================================================
# CELL 3 — GLOBAL RISK GATE TEACHER API
# =========================================================

import requests
import time
import re
import json

from google.colab import userdata


# =========================================================
# 1. API CONFIG
# =========================================================

BASE_URL = (
    "https://api.deepseek.com/chat/completions"
)

API_KEY = userdata.get(
    "DEEPSEEK"
)

MODEL = (
    "deepseek-v4-flash"
)

TEMPERATURE = 0

TIMEOUT_CONNECT = 10
TIMEOUT_READ = 45


if not API_KEY:

    raise RuntimeError(
        "DEEPSEEK API key tidak ditemukan "
        "di Colab Secrets."
    )


# =========================================================
# 2. GLOBAL RISK GATE TEACHER PROMPT
# =========================================================

TEACHER_SYSTEM_PROMPT = """
You are a strict risk-policy teacher for a security
decision model called Global Risk Gate.

You are NOT executing commands.

You are NOT an authorization mechanism.

You are ONLY assigning a preliminary risk-routing label
for training data.

The final production authority is a deterministic
Policy Engine.

Allowed labels:

SAFE
REVIEW
SANDBOX
BLOCK


=========================================================
INPUT
=========================================================

You may receive structured signals including:

- shell command or operation text
- Action Classifier probabilities
- Action Classifier predicted actions
- Command Risk probabilities
- Command Risk predicted label
- target context
- tool type
- privilege requirement
- network access
- file metadata
- scanner severity
- special deterministic rule flags


=========================================================
SAFE
=========================================================

Use SAFE when the operation is clearly low-risk and
appropriate for normal execution.

Typical characteristics:

- inspection / read-only behavior
- benign local information retrieval
- no dangerous system modification
- no destructive behavior
- no suspicious remote interaction
- no meaningful privilege escalation
- no high severity scanner finding

Examples include ordinary local status inspection,
listing files, reading non-sensitive metadata, or
similarly low-impact operations.


=========================================================
REVIEW
=========================================================

Use REVIEW when the action is not clearly safe enough
for automatic execution but is not inherently suitable
for blocking.

Typical characteristics:

- ambiguous intent or context
- unusual network activity
- writing or modifying data with uncertain scope
- elevated privilege request that may be legitimate
- suspicious classification without clear destructive
  behavior
- boundary cases requiring human or stronger policy
  inspection


=========================================================
SANDBOX
=========================================================

Use SANDBOX when execution may be useful or legitimate,
but should occur in an isolated environment.

Typical characteristics:

- unknown or untrusted code execution
- package/script execution from uncertain provenance
- security scanning or probing that should be isolated
- network-capable executable behavior with uncertain
  impact
- operations where isolation materially reduces risk
- suspicious code that still has a legitimate analysis
  use case


=========================================================
BLOCK
=========================================================

Use BLOCK when the observed behavior is sufficiently
dangerous that normal or sandbox execution should not
be automatically allowed.

Typical characteristics:

- clearly destructive behavior against meaningful data
  or system state
- severe credential or secret abuse
- strongly dangerous privilege + destructive behavior
- explicit destructive persistence or system damage
- deterministic special-rule flag requiring block
- critical scanner finding coupled with executable
  dangerous behavior


=========================================================
IMPORTANT BOUNDARIES
=========================================================

1. Global Risk Gate is preliminary risk prediction.

2. Do not assume that every WRITE is dangerous.

3. Do not assume that every NETWORK operation is
   dangerous.

4. Do not assume that every EXECUTE action must be
   blocked.

5. PRIVILEGED operations should receive increased
   scrutiny, but privilege alone does not necessarily
   mean BLOCK.

6. Destructive behavior should strongly increase risk,
   especially when combined with privilege or broad
   system scope.

7. Unknown executable code may be SANDBOX rather than
   BLOCK when isolation is appropriate.

8. Suspicious but unclear operations should normally
   prefer REVIEW over unsupported certainty.

9. If a deterministic special rule explicitly requires
   BLOCK, use BLOCK.

10. If evidence is genuinely insufficient or internally
    conflicting, set ambiguous=true.

11. Confidence represents certainty about the routing
    label, not how dangerous the operation is.

12. Base the decision on provided observable features.
    Do not invent missing behavior.


=========================================================
OUTPUT
=========================================================

Return ONLY valid JSON.

Schema:

{
  "label": "SAFE",
  "confidence": 0.95,
  "ambiguous": false
}

label:
- must be one of:
  SAFE
  REVIEW
  SANDBOX
  BLOCK

confidence:
- number from 0 to 1

ambiguous:
- true or false

Do not include explanations.
Do not include markdown.
Do not include reasoning.
"""


# =========================================================
# 3. TEACHER CALL
# =========================================================

def call_teacher(
    case,
):

    headers = {

        "Authorization":
            f"Bearer {API_KEY}",

        "Content-Type":
            "application/json",
    }


    user_payload = json.dumps(
        case,
        ensure_ascii=False,
        sort_keys=True,
    )


    payload = {

        "model":
            MODEL,

        "temperature":
            TEMPERATURE,

        "thinking": {
            "type": "disabled"
        },

        "response_format": {
            "type": "json_object"
        },

        "messages": [

            {
                "role":
                    "system",

                "content":
                    TEACHER_SYSTEM_PROMPT,
            },

            {
                "role":
                    "user",

                "content":
                    user_payload,
            },
        ],
    }


    start = (
        time.time()
    )


    response = requests.post(

        BASE_URL,

        headers=headers,

        json=payload,

        timeout=(
            TIMEOUT_CONNECT,
            TIMEOUT_READ
        ),
    )


    latency = (
        time.time()
        - start
    )


    response.raise_for_status()


    data = (
        response.json()
    )


    content = (

        data["choices"][0]
        ["message"]
        ["content"]
    )


    return (
        content,
        latency
    )


# =========================================================
# 4. PARSER
# =========================================================

def parse_teacher_output(
    text,
):

    if not text:

        raise ValueError(
            "Teacher response kosong."
        )


    text = (
        text.strip()
    )


    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I,
    )


    text = re.sub(
        r"\s*```$",
        "",
        text,
    )


    match = re.search(
        r"\{[\s\S]*?\}",
        text,
    )


    if not match:

        raise ValueError(
            "JSON tidak ditemukan: "
            + text[:300]
        )


    data = json.loads(
        match.group()
    )


    label = str(
        data.get(
            "label",
            ""
        )
    ).strip().upper()


    confidence = float(
        data.get(
            "confidence"
        )
    )


    ambiguous = data.get(
        "ambiguous"
    )


    # =====================================================
    # LABEL VALIDATION
    # =====================================================

    if label not in RISK_GATE_LABELS:

        raise ValueError(
            f"Invalid gate label: {label}"
        )


    # =====================================================
    # CONFIDENCE VALIDATION
    # =====================================================

    if not (
        0 <= confidence <= 1
    ):

        raise ValueError(
            f"Invalid confidence: "
            f"{confidence}"
        )


    # =====================================================
    # AMBIGUOUS VALIDATION
    # =====================================================

    if not isinstance(
        ambiguous,
        bool
    ):

        raise ValueError(
            "ambiguous harus boolean."
        )


    return {

        "label":
            label,

        "confidence":
            confidence,

        "ambiguous":
            ambiguous,
    }


# =========================================================
# 5. STATUS
# =========================================================

print()

print("=" * 78)
print("GLOBAL RISK GATE — TEACHER API")
print("=" * 78)

print(
    "Model           :",
    MODEL
)

print(
    "API key         :",
    "OK"
)

print(
    "Gate labels     :",
    RISK_GATE_LABELS
)

print(
    "call_teacher()  :",
    "OK"
)

print(
    "parser          :",
    "OK"
)

print()

print("=" * 78)

In [ ]:
# =========================================================
# CELL 4 — TEACHER RUNNER + RESUME + SAFE POINT
# GLOBAL RISK GATE
# =========================================================

import os
import json
import time
import random
import requests


# =========================================================
# 1. CONFIG
# =========================================================

TEACHER_MIN_CONFIDENCE = (
    0.80
)

MAX_RETRIES = (
    4
)

SAFEPOINT_EVERY = (
    10
)

REQUEST_DELAY = (
    0.20
)


# =========================================================
# 2. RETRYABLE TEACHER CALL
# =========================================================

def call_teacher_with_retry(
    case,
    max_retries=MAX_RETRIES,
):

    last_error = None


    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:

            raw_output, latency = (
                call_teacher(
                    case
                )
            )


            parsed = (
                parse_teacher_output(
                    raw_output
                )
            )


            return (
                parsed,
                raw_output,
                latency,
            )


        except (
            requests.RequestException,
            ValueError,
            KeyError,
            json.JSONDecodeError,
        ) as exc:

            last_error = exc


            if attempt >= max_retries:

                break


            wait_time = min(
                2 ** attempt,
                15,
            )


            wait_time += (
                random.random()
                * 0.5
            )


            print(
                f"Retry {attempt}/"
                f"{max_retries - 1}"
                f" — {type(exc).__name__}"
            )


            time.sleep(
                wait_time
            )


    raise RuntimeError(
        "Teacher gagal setelah retry: "
        f"{last_error}"
    )


# =========================================================
# 3. VALIDATE FEATURE CASE
# =========================================================

def validate_feature_case(
    case,
):

    if not isinstance(
        case,
        dict
    ):

        raise ValueError(
            "Feature case harus dict."
        )


    # At least specialist output should exist.

    if (
        "action_probabilities"
        not in case
    ):

        raise ValueError(
            "action_probabilities tidak ada."
        )


    if (
        "command_risk_probabilities"
        not in case
    ):

        raise ValueError(
            "command_risk_probabilities "
            "tidak ada."
        )


    return True


# =========================================================
# 4. LABEL SINGLE CASE
# =========================================================

def teacher_label_case(
    case,
):

    validate_feature_case(
        case
    )


    case_id = (
        case.get(
            "case_id"
        )
    )


    if not case_id:

        case_id = (
            make_case_id(
                case
            )
        )


    parsed, raw_output, latency = (
        call_teacher_with_retry(
            case
        )
    )


    record = {

        "case_id":
            case_id,

        "features":
            case,

        "teacher_label":
            parsed["label"],

        "teacher_confidence":
            parsed["confidence"],

        "teacher_ambiguous":
            parsed["ambiguous"],

        "teacher_model":
            MODEL,

        "teacher_latency":
            round(
                latency,
                4
            ),

        "timestamp":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
    }


    return record


# =========================================================
# 5. ACCEPTANCE POLICY
# =========================================================

def teacher_record_is_accepted(
    record,
):

    confidence = (
        record[
            "teacher_confidence"
        ]
    )

    ambiguous = (
        record[
            "teacher_ambiguous"
        ]
    )


    if ambiguous:

        return False


    if confidence < (
        TEACHER_MIN_CONFIDENCE
    ):

        return False


    return True


# =========================================================
# 6. FULL DATASET RUNNER
# =========================================================

def run_teacher_labeling(
    cases,
):

    completed_ids = (
        load_completed_case_ids()
    )


    state = (
        load_teacher_safepoint()
    )


    processed = int(
        state.get(
            "processed",
            0
        )
    )

    accepted = int(
        state.get(
            "accepted",
            0
        )
    )

    rejected = int(
        state.get(
            "rejected",
            0
        )
    )


    total = len(
        cases
    )


    print()

    print("=" * 78)
    print("GLOBAL RISK GATE — TEACHER LABELING")
    print("=" * 78)

    print(
        "Total cases       :",
        total
    )

    print(
        "Already completed :",
        len(
            completed_ids
        )
    )

    print(
        "Min confidence    :",
        TEACHER_MIN_CONFIDENCE
    )

    print(
        "Safe point every  :",
        SAFEPOINT_EVERY,
        "samples"
    )

    print()


    session_count = 0


    for index, case in enumerate(
        cases,
        start=1,
    ):

        case_id = (
            case.get(
                "case_id"
            )
        )


        if not case_id:

            case_id = (
                make_case_id(
                    case
                )
            )

            case[
                "case_id"
            ] = case_id


        # =================================================
        # RESUME SUPPORT
        # =================================================

        if case_id in completed_ids:

            continue


        try:

            record = (
                teacher_label_case(
                    case
                )
            )


            is_accepted = (
                teacher_record_is_accepted(
                    record
                )
            )


            record[
                "accepted"
            ] = is_accepted


            # =============================================
            # ALWAYS WRITE RESULT IMMEDIATELY
            # =============================================

            append_jsonl(
                TEACHER_OUTPUT_PATH,
                record,
            )


            completed_ids.add(
                case_id
            )


            processed += 1
            session_count += 1


            if is_accepted:

                accepted += 1

            else:

                rejected += 1

                append_jsonl(
                    TEACHER_REJECTED_PATH,
                    record,
                )


            print(
                f"[{index:05d}/{total:05d}] "
                f"{record['teacher_label']:<8} "
                f"conf="
                f"{record['teacher_confidence']:.3f} "
                f"amb="
                f"{record['teacher_ambiguous']} "
                f"{record['teacher_latency']:.2f}s"
            )


            # =============================================
            # PERIODIC SAFE POINT
            # =============================================

            if (
                session_count
                % SAFEPOINT_EVERY
                == 0
            ):

                save_teacher_safepoint(

                    processed=
                        processed,

                    accepted=
                        accepted,

                    rejected=
                        rejected,

                    last_case_id=
                        case_id,
                )


                print(
                    "    SAFE POINT SAVED"
                )


            time.sleep(
                REQUEST_DELAY
            )


        except Exception as exc:

            rejected += 1


            error_record = {

                "case_id":
                    case_id,

                "features":
                    case,

                "error":
                    repr(
                        exc
                    ),

                "timestamp":
                    time.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
            }


            append_jsonl(
                TEACHER_REJECTED_PATH,
                error_record,
            )


            # Save state immediately on failure.

            save_teacher_safepoint(

                processed=
                    processed,

                accepted=
                    accepted,

                rejected=
                    rejected,

                last_case_id=
                    case_id,
            )


            print(
                f"[{index:05d}/{total:05d}] "
                f"ERROR — {exc}"
            )


    # =====================================================
    # FINAL SAFE POINT
    # =====================================================

    save_teacher_safepoint(

        processed=
            processed,

        accepted=
            accepted,

        rejected=
            rejected,

        last_case_id=
            None,
    )


    print()

    print("=" * 78)
    print("TEACHER LABELING COMPLETE")
    print("=" * 78)

    print(
        "Processed :",
        processed
    )

    print(
        "Accepted  :",
        accepted
    )

    print(
        "Rejected  :",
        rejected
    )


    if processed:

        print(
            "Acceptance:",
            round(
                accepted
                / processed,
                4
            )
        )


    print()

    print(
        "Output    :",
        TEACHER_OUTPUT_PATH
    )

    print(
        "Rejected  :",
        TEACHER_REJECTED_PATH
    )

    print(
        "Safe point:",
        TEACHER_SAFEPOINT_PATH
    )

    print()

    print("=" * 78)


# =========================================================
# 7. SMOKE TEST CASE
# =========================================================

SMOKE_TEST_CASE = {

    "command":
        "cat /etc/os-release",

    "action_predictions": [
        "READ"
    ],

    "action_probabilities": {

        "READ":
            0.96,

        "WRITE":
            0.01,

        "DELETE":
            0.01,

        "EXECUTE":
            0.02,

        "NETWORK":
            0.01,

        "INSTALL":
            0.00,

        "PRIVILEGED":
            0.01,

        "SYSTEM_CHANGE":
            0.01,
    },

    "command_risk_prediction":
        "BENIGN",

    "command_risk_probabilities": {

        "BENIGN":
            0.97,

        "SUSPICIOUS":
            0.02,

        "DESTRUCTIVE":
            0.01,
    },

    "target_context":
        "local_user",

    "tool_type":
        "shell",

    "privilege_required":
        False,

    "network_access":
        False,

    "scanner_severity":
        "NONE",

    "special_rule_flags":
        [],
}


print()

print("=" * 78)
print("CELL 4 READY")
print("=" * 78)

print(
    "Runner       : run_teacher_labeling(cases)"
)

print(
    "Resume       : ENABLED"
)

print(
    "Incremental  : ENABLED"
)

print(
    "Safe point   : ENABLED"
)

print(
    "Smoke case   : READY"
)

print()

print(
    "Belum memanggil API teacher."
)

print(
    "API baru dipanggil saat "
    "teacher_label_case() atau "
    "run_teacher_labeling() dijalankan."
)

print("=" * 78)